In [1]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\sai\AppData\Local\Temp\ipykernel_23540\2719732944.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
d:\ksn-II\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""
Ingests data/telecom_guide.pdf into the 'guide' Chroma collection.
Applies RecursiveCharacterTextSplitter to break the long document into chunks.
Run once (or after regenerating the pdf ): python ingestion_pdf.ipynb
"""

"\nIngests data/telecom_guide.pdf into the 'guide' Chroma collection.\nApplies RecursiveCharacterTextSplitter to break the long document into chunks.\nRun once (or after regenerating the pdf ): python ingestion_pdf.ipynb\n"

In [3]:
CHROMA_DIR ="Chroma_store"
COLLECTION ="guides"
PDF_PATH=os.path.join("data","telecom_guide.pdf")
EMBED_MODEL="sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE=600
CHUNK_OVERLAP= 100

In [7]:
def main():
    print("Loading PDF...")
    loader= PyMuPDFLoader(PDF_PATH)
    pages=loader.load()
    print(f" {len(pages)} pages loaded.")

    print(f"Chunking (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}) ...")
    splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " "],
)
    chunks= splitter.split_documents(pages)

    #tagging chunks

    for i , chunk in enumerate(chunks):
        chunk.metadata['source']='guide'
        chunk.metadata["chunk_index"]=i

    print(f"{len(chunks)} chunks produced.")

    print("Installinf embedding model...")
    embeddings=HuggingFaceEmbeddings(model_name=EMBED_MODEL)

    print(f"Embedding and storing in chroma collection '{COLLECTION}'...")
    vectore_store=Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION,
        persist_directory=CHROMA_DIR,
    )
    print(f"Done. {vectore_store._collection.count()} vectors stored.")

In [8]:
print(main())

Loading PDF...
 9 pages loaded.
Chunking (size=600, overlap=100) ...
37 chunks produced.
Installinf embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1041.97it/s]


Embedding and storing in chroma collection 'guides'...
Done. 37 vectors stored.
None
